In [3]:
import duckdb
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [3]:
DB_USER = "postgres"
DB_PASS = "hola1234"
DB_HOST = "db-analytics.cni0hmtynfvx.us-east-1.rds.amazonaws.com"
DB_PORT = "5432"
DB_NAME = "postgres"

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

OUTPUT_DIR = "/content/drive/MyDrive/FdeA _ 2026-2/parquet_files"
OUTPUT_PATH = Path(OUTPUT_DIR)

TABLES_TO_EXPORT = [
    "products",
    "customers",
    "reps",
    "sales",
    "sale_items"
]

# 1) Crear directorio de salida
OUTPUT_PATH.mkdir(exist_ok=True)
print(f"Directorio de salida: {OUTPUT_PATH.absolute()}")

# 2) Conectar a la base de datos
engine = create_engine(DATABASE_URL, future=True)

# 3) Leer cada tabla y guardar como parquet
print(f"\nTablas a exportar: {TABLES_TO_EXPORT}\n")

for table_name in TABLES_TO_EXPORT:
    try:
        print(f"Procesando tabla: {table_name}...", end=" ")
        df = pd.read_sql_table(table_name, con=engine)

        output_file = f"{OUTPUT_DIR}/{table_name}.parquet"
        df.to_parquet(
            output_file,
            index=False,
            engine="pyarrow", # motor de archivos parquet
            compression="snappy", # formato de compresión de los datos
            use_dictionary=True, # formatear variables categóricas
            coerce_timestamps="ms", # recortar columnas de fecha-hora para que solo tengan hasta milisegundos
            use_compliant_nested_type=True # almacene de una forma correcta los datos que sean por ejemplo listas
        )

        print(f"✓ Guardado ({len(df)} filas)")
    except Exception as e:
        print(f"✗ Error: {e}")

print("\n¡Exportación completada!")

Directorio de salida: /content/drive/MyDrive/FdeA _ 2026-2/parquet_files

Tablas a exportar: ['products', 'customers', 'reps', 'sales', 'sale_items']

Procesando tabla: products... ✓ Guardado (6 filas)
Procesando tabla: customers... ✓ Guardado (10 filas)
Procesando tabla: reps... ✓ Guardado (3 filas)
Procesando tabla: sales... ✓ Guardado (20 filas)
Procesando tabla: sale_items... ✓ Guardado (20 filas)

¡Exportación completada!


In [ ]:
data_parquet = pd.read_parquet(f"{OUTPUT_DIR}/sales.parquet")
data_parquet.head()

# DWH - Datamart de ventas

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/FdeA _ 2026 -1/parquet_files"

In [ ]:
con = duckdb.connect("/content/drive/MyDrive/FdeA _ 2026 -1/dwh.duckdb")

Para validar las tablas o vistas creadas podemos usar:

In [ ]:
con.execute("""
SELECT table_schema, table_name, table_type
FROM information_schema.tables
ORDER BY table_schema, table_name
""").fetch_df()

In [ ]:
con.execute("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'sales'
""").fetch_df()

Luego debemos crear los esquemas tanto para la zona de staging como para el data warehouse

In [ ]:
con.execute("""
CREATE SCHEMA staging;
CREATE SCHEMA dw;
""")
print("Esquemas creados exitosamente!")

Para crear las tablas en la zona de staging usando los archivos debemos hacer lo siguiente

In [ ]:
con.execute(f"""
CREATE TABLE staging.products AS SELECT * FROM read_parquet('{OUTPUT_DIR}/products.parquet');
CREATE TABLE staging.customers AS SELECT * FROM read_parquet('{OUTPUT_DIR}/customers.parquet');
CREATE TABLE staging.reps AS SELECT * FROM read_parquet('{OUTPUT_DIR}/reps.parquet');
CREATE TABLE staging.sales AS SELECT * FROM read_parquet('{OUTPUT_DIR}/sales.parquet');
CREATE TABLE staging.sale_items AS SELECT * FROM read_parquet('{OUTPUT_DIR}/sale_items.parquet');
""")
print("Tablas en zona de staging creadas exitosamente!")

In [ ]:
con.execute("""
SELECT * FROM staging.products WHERE category = 'Pantallas'
""").fetch_df()

Pasemos ahora a crear las vistas de transformación. La finalidad de estas es limpiar y preparar los datos que serán insertados en las dimensiones y tablas de hechos.

In [ ]:
# una view es una consulta almacenada en la base de datos

In [ ]:
con.execute("""
SELECT * FROM staging.vw_products
""").fetch_df()

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW staging.vw_products AS
SELECT DISTINCT
  TRIM(product_id)   AS product_id,
  TRIM(UPPER(product_name)) AS product_name,
  TRIM(UPPER(category))     AS category
FROM staging.products
WHERE product_id IS NOT NULL;
""")

In [ ]:
con.execute("""
SELECT
  ROW_NUMBER() OVER (ORDER BY product_id) AS product_key,
  product_id,
  product_name,
  category
FROM staging.vw_products;
""").fetch_df()

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW staging.vw_products AS
SELECT DISTINCT
  TRIM(product_id)   AS product_id,
  TRIM(UPPER(product_name)) AS product_name,
  TRIM(category)     AS category
FROM staging.products
WHERE product_id IS NOT NULL;

CREATE VIEW staging.vw_customers AS
SELECT DISTINCT
  TRIM(customer_id)   AS customer_id,
  TRIM(customer_name) AS customer_name,
  TRIM(customer_city) AS customer_city
FROM staging.customers
WHERE customer_id IS NOT NULL;

CREATE VIEW staging.vw_reps AS
SELECT DISTINCT
  TRIM(rep_id)   AS rep_id,
  TRIM(rep_name) AS rep_name,
  TRIM(region)   AS region
FROM staging.reps
WHERE rep_id IS NOT NULL;

CREATE VIEW staging.vw_sales AS
SELECT
  sale_id,
  CAST(sale_date AS DATE) AS sale_date,
  TRIM(customer_id) AS customer_id,
  TRIM(rep_id) AS rep_id
FROM staging.sales;

CREATE VIEW staging.vw_sale_items AS
SELECT
  sale_id,
  TRIM(product_id) AS product_id,
  CAST(quantity AS INTEGER) AS quantity,
  CAST(unit_price AS DECIMAL(18,2)) AS unit_price
FROM staging.sale_items;
""")
print("Vistas de transformación y limpieza creadas exitosamente!")

Ahora si, podemos pasar a crear las dimensiones, aunque la dimensión de tiempo la crearemos de una manera diferente.

In [ ]:
con.execute("""
-- Dimensión producto
CREATE TABLE dw.dim_product (
  product_key  INTEGER,
  product_id   VARCHAR,
  product_name VARCHAR,
  category     VARCHAR
);

INSERT INTO dw.dim_product (product_key, product_id, product_name, category)
SELECT
  ROW_NUMBER() OVER (ORDER BY product_id) AS product_key,
  product_id,
  product_name,
  category
FROM staging.vw_products;

-- Luego puedes hacer:
ALTER TABLE dw.dim_product
ADD PRIMARY KEY (product_key);

-- Dimensión cliente
CREATE TABLE dw.dim_customer (
  customer_key  INTEGER,
  customer_id   VARCHAR,
  customer_name VARCHAR,
  customer_city VARCHAR
);

INSERT INTO dw.dim_customer (customer_key, customer_id, customer_name, customer_city)
SELECT
  ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key,
  customer_id,
  customer_name,
  customer_city
FROM staging.vw_customers;

ALTER TABLE dw.dim_customer
ADD PRIMARY KEY (customer_key);

-- Dimensión rep
CREATE TABLE dw.dim_rep (
  rep_key  INTEGER,
  rep_id   VARCHAR,
  rep_name VARCHAR,
  region   VARCHAR
);

INSERT INTO dw.dim_rep (rep_key, rep_id, rep_name, region)
SELECT
  ROW_NUMBER() OVER (ORDER BY rep_id) AS rep_key,
  rep_id,
  rep_name,
  region
FROM staging.vw_reps;

ALTER TABLE dw.dim_rep
ADD PRIMARY KEY (rep_key);
""")
print("Dimensiones creadas exitosamente!")

Para crear la dimensión de tiempo, podemos hacer algo como:

In [ ]:
# Obtener las fechas mínima y máxima de la tabla de ventas en DuckDB
df_sales_dates = con.execute("""
SELECT
  MIN(sale_date) AS min_date,
  MAX(sale_date) AS max_date
FROM staging.vw_sales
""").fetch_df()

min_date = df_sales_dates["min_date"].iloc[0]
max_date = df_sales_dates["max_date"].iloc[0]

dr = pd.date_range(min_date, max_date, freq="D")
df_dim_date = pd.DataFrame({"full_date": dr})

df_dim_date["year"] = df_dim_date["full_date"].dt.year
df_dim_date["month"] = df_dim_date["full_date"].dt.month
df_dim_date["day"] = df_dim_date["full_date"].dt.day
df_dim_date["day_name"] = df_dim_date["full_date"].dt.day_name()
df_dim_date["week"] = df_dim_date["full_date"].dt.isocalendar().week
df_dim_date["quarter"] = df_dim_date["full_date"].dt.quarter
df_dim_date["date_key"] = df_dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
df_dim_date = df_dim_date[[
    "date_key", "full_date", "year", "month", "day",
    "day_name", "week", "quarter"
]]

con.execute("""
CREATE TABLE dw.dim_date AS
SELECT
  date_key,
  full_date,
  year,
  month,
  day,
  day_name,
  week,
  quarter
FROM df_dim_date
""")
print("Dimensión de tiempo y fechas creada exitosamente!")

Por último, creemos la tabla de hechos de ventas:

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dw.fact_sales (
  sale_id      INTEGER,
  date_key     INTEGER,
  customer_key INTEGER,
  product_key  INTEGER,
  rep_key      INTEGER,
  quantity     INTEGER,
  unit_price   DECIMAL(18,2),
  total_sales  DECIMAL(18,2),
  PRIMARY KEY (sale_id, product_key)
);

INSERT INTO dw.fact_sales
SELECT
  si.sale_id,
  dd.date_key,
  dc.customer_key,
  dp.product_key,
  dr.rep_key,
  si.quantity,
  si.unit_price,
  si.quantity * si.unit_price
FROM staging.vw_sale_items si
JOIN staging.vw_sales s        ON si.sale_id = s.sale_id
JOIN dw.dim_date dd    ON dd.full_date = s.sale_date
JOIN dw.dim_customer dc ON dc.customer_id = s.customer_id
JOIN dw.dim_product dp  ON dp.product_id  = si.product_id
JOIN dw.dim_rep dr      ON dr.rep_id      = s.rep_id;
""")
print("Tabla de hechos creada exitosamente!!")

In [ ]:
con.execute("SELECT * FROM staging.vw_sale_items").fetchdf().head()

In [ ]:
con.execute("SELECT * FROM staging.vw_sales").fetchdf().head()

In [ ]:
con.execute("SELECT * FROM dw.fact_sales").fetchdf().head()

¿Cuál es la evolución mensual de ingresos y unidades vendidas por categoría de producto?

¿Qué clientes generan más ingresos totales?

¿Cómo se desempeñan los representantes de ventas (y por región) en términos de ingresos, número de ventas y unidades vendidas?

¿Cómo ha cambiado año con año el ingreso de una categoría específica de producto?

En un mes dado(lo puede elegir usted), ¿cuál es el promedio diario de ventas y de ingresos por cliente?

In [ ]:
con.execute("""
SELECT
  customer_name,
  SUM(total_sales) AS sum_total_sales
FROM dw.fact_sales
JOIN dw.dim_customer ON fact_sales.customer_key = dim_customer.customer_key
GROUP BY customer_name
ORDER BY sum_total_sales DESC
LIMIT 5
""").fetchdf()